<a href="https://colab.research.google.com/github/mhwang2424/pydata-book/blob/1st-edition/UNPK_%EA%B3%B5%EC%9C%A0%EC%9A%A9_Excel_%EC%88%98%EC%B9%98_%EC%82%B0%EC%B6%9C_%EC%9E%90%EB%8F%99%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. 데이터 로드 전 사전 준비

### 필수 라이브러리 호출


*   수치집계와 결과물 저장에 필요한 파이썬 라이브러리 호출




In [ ]:
pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
pip install openpyxl

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import xlsxwriter
from google.colab import files

### 일반 함수 정의


수치 산출을 위해 데이터를 가공할 함수들 정의
1.   agg_by_cri: 시트별 집계 기준(criteria)으로 KPI를 산정합니다.
2.   ls_ratio: 시트별 기준으로 산정된 수치의 Lifestyle PP의 비율을 산정합니다.
3.   calculate_total_ls_ratio: 시트별 Total 값의 Lifestyle PP 비율을 집계합니다.
4.   merge_and_create_yoy_div: 연도별 KPI 수치를 병합하고 과거 성과 대비 현재 성과로 YoY를 산출합니다.
5.   merge_and_create_yoy_subtract: 연도별 KPI 수치들을 병합하고, 현재 성과에서 과거 성과를 빼 YoY를 산출합니다 (거의 안씀).
6.   convert_tgc_for_merge_01: TGC열이 [0,1]일 때 TGC를 집계합니다.
7.   convert_tgc_for_merge_YN: TGC열이 [N,Y]일 때 TGC를 집계합니다.   
8.   recalculate_yoy: 각 KPI별 yoy를 산정합니다.
9.   portion_per_pp: 각 KPI Passionpoint 비율을 산정합니다
10.  recalculate_yoy_portion: 각 KPI의 과거 성과 대비 현재 성과로 YoY를 산출하고, 각 KPI 열 옆으로 위치를 조정합니다.
11.  convert_column_to_upper_case: 문자열을 대문자로 통일합니다.









In [ ]:
def agg_by_cri (df, criterion):
    agg_df = df.groupby(criterion).agg({"No. of Influencer" : 'nunique', "Post Volume" : 'sum', "Egmt": 'sum', "Po. Reach": 'sum'}).reset_index()
    agg_df["E/R"] = (agg_df["Egmt"] / agg_df["Po. Reach"]).astype(float)
    return agg_df

def ls_ratio (df, period, criterion):
    ls_ratio_temp = df.groupby(criterion).apply(lambda group: len(group[group['Passionpoint'] != 'Tech']) / len(group), include_groups = False)
    ratio_by_subs = ls_ratio_temp.reset_index(name = f"{period} LS Ratio")
    return ratio_by_subs

def calculate_total_ls_ratio (df, period, criterion):
    ls_ratio = len(df[df["Passionpoint"] != "Tech"]) / len(df)
    ls_ratio_df = pd.DataFrame({f'{criterion}' : ["Total"], f"{period} LS Ratio" : [ls_ratio]})
    return ls_ratio_df

def merge_and_create_yoy_div (df, df1, df2, criterion, on_col, period1, period2):
    yoy = f'YoY {on_col}'
    temp = df.merge(df1.loc[:, [criterion, on_col]], how='left', on=criterion)
    temp = temp.rename(columns = {on_col : f'{period1} {on_col}'})
    temp_final = temp.merge(df2.loc[:, [criterion, on_col]], how='left', on=criterion)
    temp_final = temp_final.rename(columns = {on_col : f'{period2} {on_col}'})
    temp_final[yoy] = temp_final[temp_final.columns[-1]] / temp_final[temp_final.columns[-2]]
    return temp_final

def merge_and_create_yoy_subtract (df, df1, df2, criterion, on_col, period1, period2):
    yoy = f'YoY {on_col}'
    temp = df.merge(df1.loc[:, [criterion, on_col]], how='left', on=criterion)
    temp = temp.rename(columns = {on_col : f'{period1} {on_col}'})
    temp_final = temp.merge(df2.loc[:, [criterion, on_col]], how='left', on=criterion)
    temp_final = temp_final.rename(columns = {on_col : f'{period2} {on_col}'})
    temp_final[yoy] = temp_final[temp_final.columns[-1]] - temp_final[temp_final.columns[-2]]
    return temp_final

def convert_tgc_for_merge_01 (df):
    # TGC 값에 따라 value 변경
    # df["TGC"] = df['TGC'].fillna(0)
    df['TGC'] = df['TGC'].replace({0: 'Non-TGC', 1: 'TGC'}) # Unpack 당시 결정되는 기준(0/1 OR N/Y)에 맞춰 변경

    return df

def convert_tgc_for_merge_YN (df):
    # TGC 값에 따라 value 변경
    # df["TGC"] = df['TGC'].fillna("N")
    df['TGC'] = df['TGC'].replace({'N': 'Non-TGC', 'Y': 'TGC'}) # Unpack 당시 결정되는 기준(0/1 OR N/Y)에 맞춰 변경

    return df

def recalculate_yoy (df, kpi_list, period1, period2):
    for i in range(len(kpi_list)):
        df.loc[:, f'YoY {kpi_list[i]}'] = df[f'{period2} {kpi_list[i]}'] / df[f'{period1} {kpi_list[i]}']
    return df

def portion_per_pp (df1, df2, kpi_list, period1, period2, col_name):
    df = pd.DataFrame({"Passionpoint" : col_name}, index=[0])
    period = [period1, period2]
    for i in range(len(kpi_list)):
        for j in range(len(period)):
            df[f'{period[j]} {kpi_list[i]}'] = df2[f'{period[j]} {kpi_list[i]}'].reset_index(drop=True) / df1[f'{period[j]} {kpi_list[i]}'].reset_index(drop=True)
    return df

def recalculate_yoy_portion (df,period1, period2, kpi_list):
    yoy_value = []
    for i in range(len(kpi_list)):
        yoy = df[f'{period2} {kpi_list[i]}'] / df[f'{period1} {kpi_list[i]}']
        yoy_value.append(yoy)

    for kpi in range(len(kpi_list)):
        col_position = df.columns.get_loc(f"{period2} {kpi_list[kpi]}") + 1
        df.insert(loc=col_position, column=f"YoY {kpi_list[kpi]}", value=yoy_value[kpi])
    return df

def convert_column_to_upper_case (df, column_name):
  df[column_name] = df[column_name].str.upper()
  return df

# TGC 결측치 보간 및 형식 일치 검수
def TGC_format_check (df, period):
  # TGC의 결측치 보정
  df["TGC"] = df["TGC"].fillna(0)
  if df["TGC"].isin([0, 1]).all():
    df["TGC"] = convert_tgc_for_merge_01(df)
    print(f"{period} TGC 형식 일치 검수 완료 \n")
  elif df["TGC"].dropna().isin(['N', 'Y']).all():
      df["TGC"] = convert_tgc_for_merge_YN(df)
      print(f"{period} TGC 형식 일치 검수 완료\n")
  else:
      print(f"{period} TGC 형식 불일치: 수기 확인 필요\n")

### 시트별 성과 집계 함수 정의

####  01. Data

In [ ]:
# 원본 데이터의 수치를 성과(KPI)로 정리하여 보여주는 함수
def overview (df, column_name):
  # 핵심 KPI
  id = ["No. of Subsidiary", "No. of Influencer", "Post Volume", "Po. Reach", "Egmt.", "E/R", "LS Infl. Portion"]
  data = []
  col = []

  # No. of Subs
  unique_subs = df["Subs"].nunique()
  data.append(unique_subs)
  # No. of Influencers
  infl_sum = df["No. of Influencer"].nunique()
  data.append(infl_sum)
  # Post Volume
  post_volume = df["Post Volume"].sum()
  data.append(post_volume)
  # Po. Reach
  po_reach = df["Po. Reach"].sum()
  data.append(po_reach)
  # Egmt
  egmt = df["Egmt"].sum()
  data.append(egmt)
  # E/R
  er = egmt / po_reach
  er = float (er)
  data.append(er)
  # LS Ratio
  ls_ratio = len(df[df["Passionpoint"] != "Tech"]) / len(df)
  data.append(ls_ratio)
  # 지정할 칼럼 이름
  col.append(column_name)

  # 결과물 테이블로 정리
  df = pd.DataFrame(data=data, index=id, columns=col)
  return df

#### 02. By Subs

In [ ]:
def subs_unique (pvt):
    temp = pvt[["Region", "Subs", "Strategic"]]
    subs_unique = temp.drop_duplicates(subset='Subs').reset_index(drop=True)
    return subs_unique

def by_subs (ori_df, pivot1, pivot2, period1, period2, criterion):
    subs = subs_unique(ori_df)
    subs_agg1 = agg_by_cri(pivot1, criterion=criterion)
    subs_agg2 = agg_by_cri(pivot2, criterion=criterion)

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(subs, subs_agg1, subs_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, subs_agg1, subs_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, subs_agg1, subs_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, subs_agg1, subs_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, subs_agg1, subs_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_subs = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_subs["YoY LS Ratio"] = final_by_subs[final_by_subs.columns[-1]].reset_index(drop=True) / final_by_subs[final_by_subs.columns[-2]].reset_index(drop=True)

    return final_by_subs

#### 03. By Reg

In [ ]:
def reg_unique (pvt):
    temp = pvt[["Region"]]
    reg_unique = temp.drop_duplicates(subset='Region').reset_index(drop=True)
    return reg_unique

def by_reg (ori_df, pivot1, pivot2, period1, period2, criterion):
    reg = reg_unique(pivot1)
    reg_agg1 = agg_by_cri(pivot1, criterion=criterion)
    reg_agg2 = agg_by_cri(pivot2, criterion=criterion)

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(reg, reg_agg1, reg_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, reg_agg1, reg_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, reg_agg1, reg_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, reg_agg1, reg_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, reg_agg1, reg_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_reg = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_reg["YoY LS Ratio"] = final_by_reg[final_by_reg.columns[-1]].reset_index(drop=True) / final_by_reg[final_by_reg.columns[-2]].reset_index(drop=True)

    return final_by_reg

#### 04. By PP

In [ ]:
def pp_unique (pvt):
    temp = pvt[["Passionpoint"]]
    pp_unique = temp.drop_duplicates(subset='Passionpoint').reset_index(drop=True)
    return pp_unique

def pp_total (pivot, criterion):
    total_temp = agg_by_cri(df=pivot, criterion=criterion)
    total = total_temp.sum()
    total_df = pd.DataFrame([total])
    total_df['E/R'] = total_df['Egmt'] / total_df['Po. Reach']
    total_df[criterion] = "Total"

    column_to_move = criterion
    columns = [column_to_move] + [col for col in total_df.columns if col != column_to_move]
    total_df = total_df[columns]

    return total_df

def total_by_pp (total_df1, total_df2, period1, period2, criterion):
    pp = total_df1[['Passionpoint']]
    pp_agg1 = total_df1
    pp_agg2 = total_df2

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(pp, pp_agg1, pp_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, pp_agg1, pp_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, pp_agg1, pp_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, pp_agg1, pp_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    final_by_pp = merge_and_create_yoy_div(egmt, pp_agg1, pp_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)

    return final_by_pp

def by_pp (current_unpack_raw, pivot1, pivot2, period1, period2, criterion):
    pp = pp_unique(current_unpack_raw)
    pp_agg1 = agg_by_cri(pivot1, criterion=criterion)
    pp_agg2 = agg_by_cri(pivot2, criterion=criterion)

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(pp, pp_agg1, pp_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, pp_agg1, pp_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, pp_agg1, pp_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, pp_agg1, pp_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, pp_agg1, pp_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 기타 전처리: Tech PP를 Table의 첫 열로 배치하여 Lifestyle인 PP 확인 용이
    tech_row = er[er["Passionpoint"] == "Tech"]
    ls_rows = er[er['Passionpoint'] != 'Tech']
    final_df_pp = pd.concat([tech_row, ls_rows]).reset_index(drop=True)

    return final_df_pp

def concat_all_by_pp (current_unpack_raw, pivot1, pivot2, period1, period2, criterion):
    by_pp_temp = by_pp (current_unpack_raw, pivot1, pivot2, period1=period1, period2=period2, criterion=criterion)
    pp_total_df1 = pp_total (pivot1, criterion=criterion)
    pp_total_df2 = pp_total (pivot2, criterion=criterion)
    total_pp = total_by_pp (pp_total_df1, pp_total_df2, period1=period1, period2=period2, criterion=criterion)
    by_pp_result = pd.concat([total_pp, by_pp_temp], ignore_index=True)
    del pp_total_df1, pp_total_df2

    return by_pp_result

#### 05. By Tier

In [ ]:
def tier_unique (pvt):
    temp = pvt[["Tier"]]
    tier_unique = temp.drop_duplicates(subset='Tier').reset_index(drop=True)
    return tier_unique

def by_tier (current_unpack_raw, pivot1, pivot2, period1, period2, criterion):
    tier = tier_unique(current_unpack_raw)
    tier_agg1 = agg_by_cri(pivot1, criterion=criterion)
    tier_agg2 = agg_by_cri(pivot2, criterion=criterion)

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(tier, tier_agg1, tier_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, tier_agg1, tier_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, tier_agg1, tier_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, tier_agg1, tier_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, tier_agg1, tier_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_tier = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_tier["YoY LS Ratio"] = final_by_tier[final_by_tier.columns[-1]].reset_index(drop=True) / final_by_tier[final_by_tier.columns[-2]].reset_index(drop=True)

    return final_by_tier

def tier_total (pivot, criterion):
    total_temp = agg_by_cri(df=pivot, criterion=criterion)
    total = total_temp.sum()
    total_df = pd.DataFrame([total])
    total_df['E/R'] = total_df['Egmt'] / total_df['Po. Reach']
    total_df[criterion] = "Total"

    column_to_move = criterion
    columns = [column_to_move] + [col for col in total_df.columns if col != column_to_move]
    total_df = total_df[columns]

    return total_df

def total_by_tier (pivot1, pivot2, total_df1, total_df2, period1, period2, criterion):
    tier = total_df1[['Tier']]
    tier_agg1 = total_df1
    tier_agg2 = total_df2

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(tier, tier_agg1, tier_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, tier_agg1, tier_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, tier_agg1, tier_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, tier_agg1, tier_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, tier_agg1, tier_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = calculate_total_ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = calculate_total_ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_tier = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_tier["YoY LS Ratio"] = final_by_tier[final_by_tier.columns[-1]].reset_index(drop=True) / final_by_tier[final_by_tier.columns[-2]].reset_index(drop=True)

    return final_by_tier

def concat_all_by_tier (current_unpack_raw, pivot1, pivot2, period1, period2, criterion):
    by_tier_temp = by_tier(current_unpack_raw, pivot1, pivot2, period1=period1, period2=period2, criterion=criterion)
    tier_total_df1 = tier_total(pivot1, criterion=criterion)
    tier_total_df2 = tier_total(pivot2, criterion=criterion)
    total_tier = total_by_tier (pivot1, pivot2, tier_total_df1, tier_total_df2, period1=period1, period2=period2, criterion=criterion)
    by_tier_result = pd.concat([total_tier, by_tier_temp], ignore_index=True)
    del tier_total_df1, tier_total_df2

    return by_tier_result

#### 06. On-Stie

In [ ]:
def tgc_dummy (ori_df):
    tgc_empty = pd.DataFrame({'TGC' : ["Total", "TGC", "Non-TGC"]})
    tgc_empty['TGC'] = tgc_empty['TGC'].astype(ori_df['TGC'].dtype)
    return tgc_empty

def by_tgc (tgc_dummy, pivot1, pivot2, period1, period2, criterion):
    tgc = tgc_dummy
    tgc_agg1 = agg_by_cri(pivot1, criterion=criterion)
    tgc_agg2 = agg_by_cri(pivot2, criterion=criterion)

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(tgc, tgc_agg1, tgc_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, tgc_agg1, tgc_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, tgc_agg1, tgc_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, tgc_agg1, tgc_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, tgc_agg1, tgc_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_tgc = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_tgc["YoY LS Ratio"] = final_by_tgc[final_by_tgc.columns[-1]].reset_index(drop=True) / final_by_tgc[final_by_tgc.columns[-2]].reset_index(drop=True)

    return final_by_tgc

def tgc_total (pivot, criterion):
    total_temp = agg_by_cri(df=pivot, criterion=criterion)
    total = total_temp.sum()
    total_df = pd.DataFrame([total])
    total_df['E/R'] = total_df['Egmt'] / total_df['Po. Reach']
    total_df[criterion] = "Total"

    column_to_move = criterion
    columns = [column_to_move] + [col for col in total_df.columns if col != column_to_move]
    total_df = total_df[columns]

    return total_df

def total_by_tgc (pivot1, pivot2, total_df1, total_df2, period1, period2, criterion):
    tgc = total_df1[['TGC']]
    tgc_agg1 = total_df1
    tgc_agg2 = total_df2

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(tgc, tgc_agg1, tgc_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, tgc_agg1, tgc_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, tgc_agg1, tgc_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, tgc_agg1, tgc_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, tgc_agg1, tgc_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = calculate_total_ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = calculate_total_ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_tgc = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_tgc["YoY LS Ratio"] = final_by_tgc[final_by_tgc.columns[-1]].reset_index(drop=True) / final_by_tgc[final_by_tgc.columns[-2]].reset_index(drop=True)

    return final_by_tgc

def concat_all_by_tgc (df_raw2, pivot1, pivot2, period1, period2, criterion):
    tgc_filter = tgc_dummy (ori_df=df_raw2)
    by_tgc_temp = by_tgc(tgc_filter, pivot1, pivot2, period1=period1, period2=period2, criterion=criterion)
    by_tgc_temp = by_tgc_temp.drop(0, axis=0)
    tgc_total_df1 = tgc_total (pivot1, criterion=criterion)
    tgc_total_df2 = tgc_total (pivot2, criterion=criterion)
    total_tgc = total_by_tgc (pivot1, pivot2, tgc_total_df1, tgc_total_df2, period1=period1, period2=period2, criterion=criterion)
    tgc_agg = pd.concat([total_tgc, by_tgc_temp], ignore_index=True)
    by_tgc_result = tgc_filter.merge(tgc_agg, how='left', on="TGC")
    del tgc_total_df1, tgc_total_df2

    return by_tgc_result

#### 7. By Product

In [ ]:
# Product별 피벗테이블 생성
def pivot_by_product (df):
    pivot = df.groupby(['Influencer UID', 'product', "Passionpoint", 'Strategic','Region', 'Subs']).agg({"Post URL" : 'nunique', "Engagements (total)": 'sum', "Potential Reach": 'sum'}).reset_index()
    pivot_product = pivot.rename(columns={"Engagements (total)" : 'Egmt', "Influencer UID" : "No. of Influencer", "Post URL" : "Post Volume", "Potential Reach" : "Po. Reach"})
    return pivot_product

# By Product별로 시점별 KPI 집계
def by_product (product_row, pivot1, pivot2, period1, period2, criterion):
    # product별 KPI 집계
    prd_agg1 = agg_by_cri(pivot1, criterion=criterion)
    prd_agg2 = agg_by_cri(pivot2, criterion=criterion)

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(product_row, prd_agg1, prd_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, prd_agg1, prd_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, prd_agg1, prd_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, prd_agg1, prd_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, prd_agg1, prd_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_prd = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_prd["YoY LS Ratio"] = final_by_prd[final_by_prd.columns[-1]].reset_index(drop=True) / final_by_prd[final_by_prd.columns[-2]].reset_index(drop=True)

    return final_by_prd

# Product Total 피벗 생성
def product_total (pivot, product_list, criterion):
    product_df = pivot[pivot[criterion].isin(product_list)]
    total_temp = agg_by_cri(df=product_df, criterion=criterion)
    total = total_temp.sum()
    total_df = pd.DataFrame([total])
    total_df['E/R'] = total_df['Egmt'] / total_df['Po. Reach']
    total_df[criterion] = "Total"

    column_to_move = criterion
    columns = [column_to_move] + [col for col in total_df.columns if col != column_to_move]
    total_df = total_df[columns]

    return total_df

# Product Total KPI 집계
def total_by_product (pivot1, pivot2, total_df1, total_df2, period1, period2, criterion):
    prd = total_df1[['product']]
    prd_agg1 = total_df1
    prd_agg2 = total_df2

    # 1. No. of Infl : Merge 이후 YoY 산정
    on_col_infl = 'No. of Influencer'
    infl_agg = merge_and_create_yoy_div(prd, prd_agg1, prd_agg2, on_col= on_col_infl, criterion=criterion, period1=period1, period2=period2)
    infl_agg = infl_agg.fillna(0)

    # 2. Post Volume : Merge 이후 YoY 산정
    on_col_post = "Post Volume"
    post_vol = merge_and_create_yoy_div(infl_agg, prd_agg1, prd_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    post_vol = post_vol.fillna(0)

    # 3. Po. Reach : Merge 이후 YoY 산정
    on_col_post = "Po. Reach"
    po_reach = merge_and_create_yoy_div(post_vol, prd_agg1, prd_agg2, on_col= on_col_post, criterion=criterion, period1=period1, period2=period2)
    po_reach = po_reach.fillna(0)

    # 4. Egmt : Merge 이후 YoY 산정
    on_col_mt = "Egmt"
    egmt = merge_and_create_yoy_div(po_reach, prd_agg1, prd_agg2, on_col = on_col_mt, criterion=criterion, period1=period1, period2=period2)
    egmt = egmt.fillna(0)

    # 5. E/R : Merge 이후 YoY 산정
    on_col_er = 'E/R'
    er = merge_and_create_yoy_div(egmt, prd_agg1, prd_agg2, on_col = on_col_er, criterion=criterion, period1=period1, period2=period2)
    er = er.fillna(0)

    # 6. LS Ratio Merge
    ls_1 = calculate_total_ls_ratio(pivot1, period1, criterion=criterion)
    ls_2 = calculate_total_ls_ratio(pivot2, period2, criterion=criterion)
    final_temp = er.merge(ls_1, how='left', on=criterion)
    final_by_prd = final_temp.merge(ls_2, how='left', on=criterion)
    final_by_prd["YoY LS Ratio"] = final_by_prd[final_by_prd.columns[-1]].reset_index(drop=True) / final_by_prd[final_by_prd.columns[-2]].reset_index(drop=True)

    return final_by_prd

# 최종 DataFrame 형성
def concat_all_by_product (product_row, pivot1, pivot2, period1, period2, product_list, criterion):
    by_prd_temp = by_product(product_row, pivot1, pivot2, period1=period1, period2=period2, criterion=criterion)
    prd_total_df1 = product_total(pivot1, product_list, criterion=criterion)
    prd_total_df2 = product_total(pivot2, product_list, criterion=criterion)
    total_tier = total_by_product (pivot1, pivot2, prd_total_df1, prd_total_df2, period1=period1, period2=period2, criterion=criterion)
    by_prd_result = pd.concat([total_tier, by_prd_temp], ignore_index=True)
    del prd_total_df1, prd_total_df2

    return by_prd_result

# 1. 데이터 로드

파일 업로드 순서: 파일 선택의 버튼 클릭 후 하기 순서로 해당 파일들 차례로 업로드

1. 과거 시점의 Raw Data

2. 최근 시점의 Raw Data

3. Subs별 E/R KPI 타겟 시트 [Region - Subs -Strategic - E/R KPI Target 칼럼 순서의 엑셀 시트]

In [ ]:
# 이전 시점 데이터
uploaded1 = files.upload()
# 최근 시점 데이터
uploaded2 = files.upload()
#  E/R KPI Target 참조 데이터
uploaded3 = files.upload()
# 이전 시점 데이터의 파일명
file_name1 = list(uploaded1.keys())[0]
# 최근 시점 데이터의 파일명
file_name2 = list(uploaded2.keys())[0]
# E/R KPI Target 참조 데이터의 파일명
file_name3 = list(uploaded3.keys())[0]

sheet_unq = 'post_uniq'  # Post Uniq 시트명
sheet_prd = 'post_by_product' # Post By Product 시트명

# 데이터 로드 및 전처리
# 이전 시점 데이터 로드
df_raw_prior = pd.read_excel(file_name1, sheet_name = sheet_unq)
df_raw_prior = convert_tgc_for_merge_YN (df_raw_prior)
df_raw_prd1 = pd.read_excel(file_name1, sheet_name = sheet_prd)

# 최근 시점 데이터 로드
df_raw_recent = pd.read_excel(file_name2, sheet_name = sheet_unq)
df_raw_recent = convert_tgc_for_merge_YN (df_raw_recent)
df_raw_prd2 = pd.read_excel(file_name2, sheet_name = sheet_prd)

# KPI 정의
kpi_list = ['No. of Influencer', 'Post Volume', "Po. Reach", "Egmt", "E/R"]

# Subs별 E/R Target 참조 파일 [희망하는 양식에 부합하는 엑셀 파일 로드 필수: Region - Subs -Strategic - E/R KPI Target 칼럼 순서의 파일 함께 업로드 필수]
er_kpi_target_df = pd.read_excel(file_name3)

print("데이터 로드 완료")

Saving KS4C44~1.XLS to KS4C44~1.XLS


Saving Samsung_175943_Unpacked_unpacked_1W_250130+1511_250130+1513_UNIQUE(ds).xlsx to Samsung_175943_Unpacked_unpacked_1W_250130+1511_250130+1513_UNIQUE(ds).xlsx


Saving Subs별 ER KPI Target vF.xlsx to Subs별 ER KPI Target vF.xlsx
데이터 로드 완료


# 2. 사용자 지정
1. 비교 기간 설정
*   period1: YoY 전년도 Unpack 시점 입력 (e.g. 24 UNPK 2D 1W)
*   period2: YoY 올해 Unpack 시점 입력 (e.g. 25 UNPK 2D 1W)



2. 성과 출력하고자 희망하는 제품군 설정
*   product_list:  대괄호 내 성과 산정하고자 하는 제품군 입력 (e.g. "Z", "Buds", "Edge" ...)
* 전체 제품의 성과 출력하고자 하는 경우:  "df_raw_prd2['product'].unique()" 로 입력 (df_raw_prd2 = 금번 Unpack 데이터의 제품 Raw Data)







In [ ]:
# 비교 기간 설정
period1 = '24 UNPK 1H 1W'   # 과거 시점 기입
period2 = '25 UNPK 1H 1W'   # 최근 시점 기입

# By Product 제품군 지정
product_list = df_raw_prd2['product'].unique()

# 3. Filter Format (Story & Feed or Feed Only)

#### Feed Only 성과 산출

*   Story 포스팅까지 포함한 All 성과 집계 시 Skip



In [ ]:
# Feed Only
filer_format = "story"
filter_format_case_standardized = filer_format.casefold()
df_raw_prior['format'] = df_raw_prior['format'].fillna('').str.casefold()
df_raw_prior["format"] = df_raw_prior['format'].str.casefold()
df_raw_recent['format'] = df_raw_recent['format'].fillna('').str.casefold()
df_raw_recent['format'] = df_raw_recent["format"].str.casefold()
df_raw1 = df_raw_prior[df_raw_prior["format"] != filter_format_case_standardized]
df_raw2 = df_raw_recent[df_raw_recent["format"] != filter_format_case_standardized]

df_raw_prd1 = df_raw_prd1[df_raw_prd1["format"] != filter_format_case_standardized]
df_raw_prd2 = df_raw_prd2[df_raw_prd2["format"] != filter_format_case_standardized]

final_file_name = 'Feed Only'

####  All 성과 산출

*   Story 포스팅은 제외한 Feed Only 성과 집계 시 Skip

In [ ]:
#  All
df_raw1 = df_raw_prior
df_raw2 = df_raw_recent
final_file_name = 'All'

# 4. 전체 실행

0. Raw Data 피벗 테이블 형성
*   pivot1: YoY 전년도 Raw Data의 Unpack 결과 집계 기준과 KPI별 Pivot Table 생성
*   pivot2: YoY 올해(이번 Unpack 기간) Raw Data의 Unpack 결과 집계 기준과 KPI별 Pivot Table 생성

*   Unpack 결과 집계 기준: "Region", "Subs", "Strategic", "Passionpoint", "Tier", "TGC"
*   KPI: "No. of Influencer", "Post Volume", "Engagement", "Potential Reach", "E/R"







In [ ]:
# 원본 데이터의 피벗 테이블
pivot1 = df_raw1.groupby(["Influencer UID", "Region", "Subs", "Strategic", "Passionpoint", "Tier", "TGC"]).agg({"Post URL" : 'nunique', "Engagements (total)": 'sum', "Potential Reach": 'sum'}).reset_index()
pivot1 = pivot1.rename(columns={"Engagements (total)" : 'Egmt', "Influencer UID" : "No. of Influencer", "Post URL" : "Post Volume", "Potential Reach" : "Po. Reach"})

pivot2 = df_raw2.groupby(["Influencer UID", "Region", "Subs", "Strategic", "Passionpoint", "Tier", "TGC"]).agg({"Post URL" : 'nunique', "Engagements (total)": 'sum', "Potential Reach": 'sum'}).reset_index()
pivot2 = pivot2.rename(columns={"Engagements (total)" : 'Egmt', "Influencer UID" : "No. of Influencer", "Post URL" : "Post Volume", "Potential Reach" : "Po. Reach"})

1. Data Summary Sheet 저장

*   총 KPI 성과 산정한 결과 Data 시트로 집계



In [ ]:
print(f"{period2} {final_file_name} 1. Data Sheet 작성 시작")
# 시점별 원본데이터 전체 KPI 집계
Data_1w = overview(pivot1, column_name = period1)
Data_2w = overview(pivot2, column_name = period2)
Data_final = pd.merge(Data_1w, Data_2w, left_index=True, right_index=True, how='inner')
Data_final["YoY_Data"] = Data_final[Data_final.columns[-1]] / Data_final[Data_final.columns[-2]]
# 변수 삭제로 메모리 용량 확보
del Data_1w, Data_2w
print(f"{period2} {final_file_name} 1. Data Sheet 작성 완료\n")

25 UNPK 1H 1W Feed Only 1. Data Sheet 작성 시작
25 UNPK 1H 1W Feed Only 1. Data Sheet 작성 완료



2. By Subs Sheet 저장

*   법인별 KPI 성과 산정한 결과 By Subs 시트로 집계
*   사전에 로드한 "법인별 E/R KPI Target" 파일과 매핑하여 목표 대비 실제 성과 비교



In [ ]:
print(f"{period2} {final_file_name} 2. By Subs Sheet 작성 시작")
# KPI 집계 기준
cri_subs = "Subs"
pivot1 = convert_column_to_upper_case(pivot1, "Subs")
pivot2 = convert_column_to_upper_case(pivot2, "Subs")
by_subs_result = by_subs(er_kpi_target_df, pivot1, pivot2, period1=period1, period2=period2, criterion=cri_subs)

# E/R KPI Target, vs. Target 칼럼 추가
er_kpi_target_column_name = "E/R KPI 타겟*" # 엑셀 상의 칼럼명 바뀔 시 수정 요망
er_kpi_target_df = convert_column_to_upper_case(er_kpi_target_df, "Subs")

print("KPI Target Subs와 현재 데이터 형식 일치 검수 완료")
by_subs_result_sorted = er_kpi_target_df.merge(by_subs_result, on=['Region', 'Subs', "Strategic"], how='left')
insert_loc = by_subs_result_sorted.columns.get_loc("YoY E/R") + 1
by_subs_result_sorted.insert(loc=insert_loc, column = "vs. Target" , value = by_subs_result_sorted[f'{period2} E/R'].reset_index(drop=True) / by_subs_result_sorted[er_kpi_target_column_name].reset_index(drop=True))
print(f"{period2} {final_file_name} 2. By Subs Sheet 작성 완료\n")

25 UNPK 1H 1W Feed Only 2. By Subs Sheet 작성 시작
KPI Target Subs와 현재 데이터 형식 일치 검수 완료
25 UNPK 1H 1W Feed Only 2. By Subs Sheet 작성 완료



3. By Region Sheet 저장

*   권역별 KPI 성과 산정한 결과 By Region 시트로 집계



In [ ]:
print(f"{period2} {final_file_name} 3. By Region Sheet 작성 시작")
# KPI 집계 기준
cri_reg = 'Region'
by_reg_result = by_reg(er_kpi_target_df, pivot1, pivot2, period1=period1, period2=period2, criterion=cri_reg)
print(f"{period2} {final_file_name} 3. By Region Sheet 작성 완료\n")

25 UNPK 1H 1W Feed Only 3. By Region Sheet 작성 시작
25 UNPK 1H 1W Feed Only 3. By Region Sheet 작성 완료



4. By Passionpoint Sheet 저장

*   Passionpoint별 KPI 성과 산정한 결과 By Passionpoint 시트로 집계
*   Tech/Lifestyle 절대값 = Tech PP와 Tech 외 Lifestyle PP 간 실제 수치
*   Tech/Lifestyle 비중 = Total 대비 Tech PP와 Tech 외 Lifestyle PP 비율





In [ ]:
print(f"{period2} {final_file_name} 4. By Passionpoint Sheet 작성 시작")
# KPI 집계 기준
cri_pp = "Passionpoint"
by_pp_result = concat_all_by_pp (current_unpack_raw=df_raw_recent, pivot1=pivot1, pivot2=pivot2, period1=period1, period2=period2, criterion=cri_pp)
# 4-1. PP별 절대값 집계
# Total
total_sum_pp = by_pp_result[by_pp_result["Passionpoint"] == "Total"]
# Tech PP
tech_sum = by_pp_result[by_pp_result["Passionpoint"] == "Tech"]
tech_sum.loc[:, "Passionpoint"] = "Tech 절대값"
# Lifestyle PP
ls_sum = by_pp_result[(by_pp_result["Passionpoint"] != "Tech") & (by_pp_result["Passionpoint"] != "Total")]
ls_total = ls_sum.sum()
# Lifestyle PP 전체 E/R 재조정
ls_total[f'{period1} E/R'] = ls_total[f'{period1} Egmt'] / ls_total[f'{period1} Po. Reach']
ls_total[f'{period2} E/R'] = ls_total[f'{period2} Egmt'] / ls_total[f'{period2} Po. Reach']
# 데이터프레임화
ls_total_df = pd.DataFrame([ls_total])
ls_total_df.loc[:, "Passionpoint"] = "Lifestyle 절대값"
ls_sum_final = recalculate_yoy (ls_total_df, kpi_list=kpi_list, period1=period1, period2=period2)
# Total + Tech + Lifestyle 병합
pp_sum = pd.concat([tech_sum, ls_sum_final], ignore_index = True)

# 4-2. PP별 비중 집계
tech_pp_portion = portion_per_pp (df1=total_sum_pp, df2=tech_sum, kpi_list=kpi_list, period1=period1, period2=period2, col_name="Tech 비중")
ls_pp_portion = portion_per_pp (df1=total_sum_pp, df2=ls_sum_final, kpi_list=kpi_list, period1=period1, period2=period2, col_name="Lifestyle 비중")

# 집계된 값에 맞게 YoY 조정
ls_pp_portion_final = recalculate_yoy_portion (ls_pp_portion,period1, period2, kpi_list)
tech_pp_portion_final = recalculate_yoy_portion (tech_pp_portion,period1, period2, kpi_list)

# 절대값과 비중 Merge 한 최종 결과 생성
pp_portion_df = pd.concat([tech_pp_portion_final, ls_pp_portion_final], ignore_index=True)
pp_agg = pd.concat([pp_sum, pp_portion_df], ignore_index=True)
pp_final_df = pd.concat([by_pp_result, pp_agg], ignore_index=True)
print(f"{period2} {final_file_name} 4. By Passionpoint Sheet 작성 완료\n")

25 UNPK 1H 1W Feed Only 4. By Passionpoint Sheet 작성 시작
25 UNPK 1H 1W Feed Only 4. By Passionpoint Sheet 작성 완료



5. By Tier Sheet 저장

*   Tier별 KPI 성과 산정한 결과 By Region 시트로 집계



In [ ]:
print(f"{period2} {final_file_name} 5. By Tier Sheet 작성 시작")
# KPI 집계 기준
cri_tier = 'Tier'
by_tier_result = concat_all_by_tier (current_unpack_raw=df_raw_recent, pivot1=pivot1, pivot2=pivot2, period1=period1, period2=period2, criterion=cri_tier)
print(f"{period2} {final_file_name} 5. By Tier Sheet 작성 완료\n")

25 UNPK 1H 1W Feed Only 5. By Tier Sheet 작성 시작
25 UNPK 1H 1W Feed Only 5. By Tier Sheet 작성 완료



6. On-Site Sheet 저장

*   TGC vs. Non-TGC별 KPI 성과 산정한 결과 On-Site 시트로 집계



In [ ]:
print(f"{period2} {final_file_name} 6. On-Site Sheet 작성 시작")
# KPI 집계 기준
cri_tgc = 'TGC'
by_tgc_result = concat_all_by_tgc (df_raw_recent, pivot1, pivot2, period1, period2, criterion=cri_tgc)
by_tgc_result = by_tgc_result.rename(columns = {"TGC" : "Type"})
print(f"{period2} {final_file_name} 6. On-Site Sheet 작성 완료\n")

25 UNPK 1H 1W Feed Only 6. On-Site Sheet 작성 시작
25 UNPK 1H 1W Feed Only 6. On-Site Sheet 작성 완료



7. By Product Sheet 저장


*   금번 Unpack으로 출시된 기기 모델별 KPI 성과 산정하여 By Product 시트로 집계



In [ ]:
print(f"{period2} {final_file_name} 7. By Product Sheet 작성 시작")
# 사용자가 지정한 제품군만으로 분석 진행
product_row = pd.DataFrame({'product': product_list})
# KPI 집계 기준
cri_prd = 'product'
pivot_prd_1 = pivot_by_product (df_raw_prd1)
pivot_prd_2 = pivot_by_product (df_raw_prd2)
by_product_result = concat_all_by_product (product_row, pivot1=pivot_prd_1, pivot2=pivot_prd_2, period1=period1, period2=period2, product_list=product_list, criterion=cri_prd)
print(f"{period2} {final_file_name} 7. By Product Sheet 작성 완료\n")

print(f"{period2} {final_file_name} 전체 Excel 수치 산출 완료")

25 UNPK 1H 1W Feed Only 7. By Product Sheet 작성 시작
25 UNPK 1H 1W Feed Only 7. By Product Sheet 작성 완료

25 UNPK 1H 1W Feed Only 전체 Excel 수치 산출 완료


# 5. 최종 결과물 저장


*   저장한 시트별 결과물 xlsx 파일 format으로 출력



In [ ]:
save_sheet_data = '1.Data'
save_sheet_subs = '2.By Subs'
save_sheet_region = '3.By Region'
save_sheet_pp = '4.By Passionpoint'
save_sheet_tier = '5.By Tier'
save_sheet_on_site = "6.On-Site"
save_sheet_product = '7.By Product'

# 최종 결과물을 엑셀 파일 포맷 (.xlsx)로 저장하게 해주는 함수
def save_file (final_file_name, current_unpack_period):
  with pd.ExcelWriter(f'{current_unpack_period} Unpack {final_file_name} 결과.xlsx', engine='xlsxwriter') as writer:
    Data_final.to_excel(writer, sheet_name=f'{save_sheet_data} {final_file_name}', index=True)
    by_subs_result_sorted.to_excel(writer, sheet_name=f'{save_sheet_subs} {final_file_name}', index=False)
    by_reg_result.to_excel(writer, sheet_name=f'{save_sheet_region} {final_file_name}', index=False)
    pp_final_df.to_excel(writer, sheet_name=f'{save_sheet_pp} {final_file_name}', index=False)
    by_tier_result.to_excel(writer, sheet_name=f'{save_sheet_tier} {final_file_name}', index=False)
    by_tgc_result.to_excel(writer, sheet_name=f'{save_sheet_on_site} {final_file_name}', index=False)
    by_product_result.to_excel(writer, sheet_name=f'{save_sheet_product} {final_file_name}', index=False)
    print(f'{final_file_name} 파일 저장 완료')

# 실행 후 아래 'Feed Only 또는 All 파일 저장 완료'라는 메세지 확인 후 좌상단 파일 아이콘 클릭으로 산출물 생성되었는 지 확인 후, 개별 로컬 환경으로 수동 설치
save_file (final_file_name=final_file_name, current_unpack_period=period2)

Feed Only 파일 저장 완료
